In [1]:
"""
Evolution of Deep Convolutional Neural Networks — Part 2 Implementation.

This module provides a modular, parameterized PyTorch implementation of the
two architectural building blocks that most influenced the CNN lineage
studied in Part 1 of this assignment:

    1. VGGBlock       -- a stack of N same-padded k x k convolutions
                          (optionally batch-normalized) followed by a single
                          spatial down-sampling (max-pool) step, as
                          popularised by Simonyan & Zisserman's VGGNet [3].

    2. ResidualBlock   -- a "basic" (2 x 3x3) or "bottleneck" (1x1-3x3-1x1)
                          residual unit with an identity/projection shortcut,
                          as introduced by He et al. in ResNet [5].

Both blocks are fully parameterized (channel widths, depth, stride, kernel
size, normalization) so that they can be instantiated as structural
*variants* rather than hard-coded for one specific network. Each block
exposes:

    * an analytical parameter-count formula (`.analytical_param_count()`)
      that is checked against PyTorch's own `numel()`-based count, and
    * a `forward()` pass that is wrapped by a shared `ShapeTracker` utility
      so that the exact tensor dimensions at every internal layer are
      recorded and can be printed as a table.

"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import List, Optional

import torch
import torch.nn as nn


# --------------------------------------------------------------------------- #
# Utility: shape / parameter tracking
# --------------------------------------------------------------------------- #

@dataclass
class LayerRecord:
    """A single row of the per-layer trace produced while a block runs."""
    name: str
    layer_type: str
    output_shape: torch.Size
    n_params: int


@dataclass
class ShapeTracker:
    """Collects LayerRecord rows as a block/network executes.

    Any block in this module accepts an optional `tracker` argument in its
    `forward()` method; when supplied, every internal sub-layer appends a
    record of its name, type, output tensor shape, and parameter count.
    """
    records: List[LayerRecord] = field(default_factory=list)

    def log(self, name: str, module: nn.Module, output: torch.Tensor) -> None:
        n_params = sum(p.numel() for p in module.parameters())
        self.records.append(
            LayerRecord(
                name=name,
                layer_type=module.__class__.__name__,
                output_shape=tuple(output.shape),
                n_params=n_params,
            )
        )

    def print_table(self, title: str = "Layer Trace") -> None:
        print(f"\n{'-' * 78}\n{title}\n{'-' * 78}")
        header = f"{'Layer':<28}{'Type':<14}{'Output Shape':<24}{'Params':>10}"
        print(header)
        print("-" * 78)
        total = 0
        for r in self.records:
            print(f"{r.name:<28}{r.layer_type:<14}{str(r.output_shape):<24}{r.n_params:>10,}")
            total += r.n_params
        print("-" * 78)
        print(f"{'TOTAL':<66}{total:>10,}")
        print("-" * 78)


def count_parameters(module: nn.Module) -> int:
    """Exact PyTorch parameter count (trainable + frozen)."""
    return sum(p.numel() for p in module.parameters())


def conv_out_size(in_size: int, kernel: int, stride: int, padding: int) -> int:
    """Standard convolution/pooling output-size formula (square inputs)."""
    return (in_size + 2 * padding - kernel) // stride + 1


# --------------------------------------------------------------------------- #
# Block 1: VGG Block
# --------------------------------------------------------------------------- #

class VGGBlock(nn.Module):
    """A parameterized VGG-style convolutional block.

    Implements the design principle introduced in VGGNet [3]: replace large
    receptive-field filters with a *stack* of small (default 3x3), stride-1,
    same-padded convolutions. Stacking `num_convs` such layers before a
    single 2x2 max-pool gives the same effective receptive field as one
    large-kernel convolution while using fewer parameters and adding more
    non-linearities (one ReLU per conv instead of one per large filter).

    Parameters
    ----------
    in_channels : int
        Number of input feature-map channels.
    out_channels : int
        Number of output channels produced by every conv layer in the block.
    num_convs : int, default 2
        Number of stacked convolution+activation layers before pooling.
    kernel_size : int, default 3
        Spatial size of the square convolution kernel.
    use_batchnorm : bool, default True
        Whether to insert BatchNorm2d after every convolution (a later
        refinement not present in the original 2014 VGG paper but standard
        practice today, included here as a structural variant switch).
    pool : bool, default True
        Whether to apply a 2x2 stride-2 max-pool at the end of the block.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        num_convs: int = 2,
        kernel_size: int = 3,
        use_batchnorm: bool = True,
        pool: bool = True,
    ) -> None:
        super().__init__()
        if num_convs < 1:
            raise ValueError("num_convs must be >= 1")
        if kernel_size % 2 == 0:
            raise ValueError("kernel_size must be odd to preserve spatial size ('same' padding)")

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_convs = num_convs
        self.kernel_size = kernel_size
        self.use_batchnorm = use_batchnorm
        self.pool = pool
        padding = kernel_size // 2  # 'same' padding for stride-1 convs

        layers: List[nn.Module] = []
        c_in = in_channels
        for i in range(num_convs):
            layers.append(nn.Conv2d(c_in, out_channels, kernel_size, stride=1, padding=padding, bias=not use_batchnorm))
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.ReLU(inplace=True))
            c_in = out_channels
        self.convs = nn.ModuleList(layers)

        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2) if pool else None

    def analytical_param_count(self) -> int:
        """Closed-form parameter count, cross-checked against PyTorch below."""
        total = 0
        c_in = self.in_channels
        for _ in range(self.num_convs):
            conv_params = (self.kernel_size ** 2) * c_in * self.out_channels
            conv_params += 0 if self.use_batchnorm else self.out_channels  # conv bias
            bn_params = 2 * self.out_channels if self.use_batchnorm else 0  # gamma, beta
            total += conv_params + bn_params
            c_in = self.out_channels
        return total

    def forward(self, x: torch.Tensor, tracker: Optional[ShapeTracker] = None) -> torch.Tensor:
        conv_idx = 0
        for layer in self.convs:
            x = layer(x)
            if tracker is not None and isinstance(layer, nn.Conv2d):
                conv_idx += 1
                tracker.log(f"VGGBlock.conv{conv_idx}", layer, x)
        if self.maxpool is not None:
            x = self.maxpool(x)
            if tracker is not None:
                tracker.log("VGGBlock.maxpool", self.maxpool, x)
        return x


# --------------------------------------------------------------------------- #
# Block 2: Residual Block (ResNet)
# --------------------------------------------------------------------------- #

class ResidualBlock(nn.Module):
    """A parameterized residual block, in 'basic' or 'bottleneck' form.

    Implements the core idea of ResNet [5]: instead of learning a direct
    mapping H(x), each block learns a residual F(x) = H(x) - x, added back
    to the input via a shortcut connection: output = ReLU(F(x) + shortcut(x)).
    This reformulation keeps gradients flowing through the identity path
    during backpropagation, which allows networks hundreds of layers deep to
    be trained without the degradation/vanishing-gradient problem that
    limited plain stacks of convolutions.

    Two structural variants are supported:

    * basic block      (bottleneck=False): two 3x3 convolutions. Used in
      ResNet-18/34.
    * bottleneck block (bottleneck=True):  1x1 (reduce) -> 3x3 -> 1x1
      (expand, x4 channels) convolutions. Used in ResNet-50/101/152; the
      1x1 convolutions cut the parameter/FLOP cost of the expensive 3x3
      convolution by shrinking its channel width first and restoring it
      afterward, which is the "bottleneck" efficiency trick.

    Parameters
    ----------
    in_channels : int
        Number of input channels.
    out_channels : int
        For a basic block: the number of output channels.
        For a bottleneck block: the number of *reduced* (bottleneck) channels;
        the block's true output width is `out_channels * expansion`.
    stride : int, default 1
        Stride applied in the first (basic) or middle 3x3 (bottleneck) conv.
        stride=2 is used when a block also performs spatial down-sampling.
    bottleneck : bool, default False
        Selects the basic (2-conv) or bottleneck (3-conv) structural variant.
    expansion : int, default 4
        Channel-expansion factor applied at the end of a bottleneck block.
        Ignored when bottleneck=False (expansion is always 1 in that case).
    use_batchnorm : bool, default True
        Whether to apply BatchNorm2d after every convolution.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        bottleneck: bool = False,
        expansion: int = 4,
        use_batchnorm: bool = True,
    ) -> None:
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride
        self.bottleneck = bottleneck
        self.expansion = expansion if bottleneck else 1
        self.use_batchnorm = use_batchnorm
        final_channels = out_channels * self.expansion

        def bn(c):
            return nn.BatchNorm2d(c) if use_batchnorm else nn.Identity()

        if not bottleneck:
            # Basic block: 3x3 -> 3x3
            self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=not use_batchnorm)
            self.bn1 = bn(out_channels)
            self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=not use_batchnorm)
            self.bn2 = bn(out_channels)
            self.conv3 = None
            self.bn3 = None
        else:
            # Bottleneck block: 1x1 (reduce) -> 3x3 -> 1x1 (expand)
            self.conv1 = nn.Conv2d(in_channels, out_channels, 1, stride=1, padding=0, bias=not use_batchnorm)
            self.bn1 = bn(out_channels)
            self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=stride, padding=1, bias=not use_batchnorm)
            self.bn2 = bn(out_channels)
            self.conv3 = nn.Conv2d(out_channels, final_channels, 1, stride=1, padding=0, bias=not use_batchnorm)
            self.bn3 = bn(final_channels)

        self.relu = nn.ReLU(inplace=True)

        # Projection shortcut is required whenever the shortcut path must
        # change either spatial resolution (stride != 1) or channel width
        # (in_channels != final_channels); otherwise the identity is used
        # directly, exactly as specified in the original ResNet paper.
        needs_projection = (stride != 1) or (in_channels != final_channels)
        if needs_projection:
            self.shortcut_conv = nn.Conv2d(in_channels, final_channels, 1, stride=stride, padding=0, bias=not use_batchnorm)
            self.shortcut_bn = bn(final_channels)
        else:
            self.shortcut_conv = None
            self.shortcut_bn = None

    def analytical_param_count(self) -> int:
        """Closed-form parameter count, cross-checked against PyTorch below."""

        def conv_p(k, cin, cout):
            p = k * k * cin * cout
            p += 0 if self.use_batchnorm else cout
            return p

        def bn_p(c):
            return 2 * c if self.use_batchnorm else 0

        total = 0
        if not self.bottleneck:
            total += conv_p(3, self.in_channels, self.out_channels) + bn_p(self.out_channels)
            total += conv_p(3, self.out_channels, self.out_channels) + bn_p(self.out_channels)
        else:
            final_channels = self.out_channels * self.expansion
            total += conv_p(1, self.in_channels, self.out_channels) + bn_p(self.out_channels)
            total += conv_p(3, self.out_channels, self.out_channels) + bn_p(self.out_channels)
            total += conv_p(1, self.out_channels, final_channels) + bn_p(final_channels)

        if self.shortcut_conv is not None:
            final_channels = self.out_channels * self.expansion
            total += conv_p(1, self.in_channels, final_channels) + bn_p(final_channels)

        return total

    def forward(self, x: torch.Tensor, tracker: Optional[ShapeTracker] = None) -> torch.Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        if tracker is not None:
            tracker.log("ResBlock.conv1", self.conv1, out)

        out = self.conv2(out)
        out = self.bn2(out)
        if self.conv3 is not None:
            out = self.relu(out)
        if tracker is not None:
            tracker.log("ResBlock.conv2", self.conv2, out)

        if self.conv3 is not None:
            out = self.conv3(out)
            out = self.bn3(out)
            if tracker is not None:
                tracker.log("ResBlock.conv3", self.conv3, out)

        if self.shortcut_conv is not None:
            identity = self.shortcut_conv(x)
            identity = self.shortcut_bn(identity)
            if tracker is not None:
                tracker.log("ResBlock.shortcut", self.shortcut_conv, identity)

        out = out + identity           # the residual addition (element-wise)
        out = self.relu(out)
        return out


# --------------------------------------------------------------------------- #
# Demonstration networks built purely from the two modular blocks above
# --------------------------------------------------------------------------- #

class TinyVGG(nn.Module):
    """A small VGG-style classifier assembled from stacked VGGBlocks.

    Structural variant used for demonstration: 4 VGGBlocks with 2 stacked
    3x3 convs each, doubling channel width every block, followed by global
    average pooling and a linear classifier head.
    """

    def __init__(self, in_channels: int = 3, num_classes: int = 10, base_width: int = 32) -> None:
        super().__init__()
        widths = [base_width, base_width * 2, base_width * 4, base_width * 8]
        blocks = []
        c_in = in_channels
        for w in widths:
            blocks.append(VGGBlock(c_in, w, num_convs=2, kernel_size=3, use_batchnorm=True, pool=True))
            c_in = w
        self.blocks = nn.ModuleList(blocks)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(widths[-1], num_classes)

    def forward(self, x: torch.Tensor, tracker: Optional[ShapeTracker] = None) -> torch.Tensor:
        for i, block in enumerate(self.blocks, start=1):
            x = block(x, tracker=tracker)
            if tracker is not None:
                tracker.log(f"[after VGGBlock {i}]", nn.Identity(), x)
        x = self.gap(x)
        if tracker is not None:
            tracker.log("GlobalAvgPool", self.gap, x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        if tracker is not None:
            tracker.log("Linear(fc)", self.fc, x)
        return x


class TinyResNet(nn.Module):
    """A small ResNet-style classifier assembled from stacked ResidualBlocks.

    Structural variant used for demonstration: a 7x7 stem convolution
    followed by 4 stages of bottleneck ResidualBlocks (2 blocks per stage),
    the first block of each stage down-sampling with stride 2, followed by
    global average pooling and a linear classifier head -- the same overall
    template used by ResNet-50/101/152 [5].
    """

    def __init__(self, in_channels: int = 3, num_classes: int = 10, base_width: int = 16, bottleneck: bool = True) -> None:
        super().__init__()
        self.stem_conv = nn.Conv2d(in_channels, base_width, kernel_size=7, stride=2, padding=3, bias=False)
        self.stem_bn = nn.BatchNorm2d(base_width)
        self.stem_relu = nn.ReLU(inplace=True)
        self.stem_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        expansion = 4 if bottleneck else 1
        stage_widths = [base_width, base_width * 2, base_width * 4, base_width * 8]
        stages = []
        c_in = base_width
        for stage_idx, w in enumerate(stage_widths):
            stride = 1 if stage_idx == 0 else 2
            b1 = ResidualBlock(c_in, w, stride=stride, bottleneck=bottleneck, expansion=expansion)
            b2 = ResidualBlock(w * expansion, w, stride=1, bottleneck=bottleneck, expansion=expansion)
            stages.append(nn.ModuleList([b1, b2]))
            c_in = w * expansion
        self.stages = nn.ModuleList(stages)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(c_in, num_classes)

    def forward(self, x: torch.Tensor, tracker: Optional[ShapeTracker] = None) -> torch.Tensor:
        x = self.stem_conv(x)
        x = self.stem_bn(x)
        x = self.stem_relu(x)
        if tracker is not None:
            tracker.log("Stem.conv7x7", self.stem_conv, x)
        x = self.stem_pool(x)
        if tracker is not None:
            tracker.log("Stem.maxpool", self.stem_pool, x)

        for s_idx, stage in enumerate(self.stages, start=1):
            for b_idx, block in enumerate(stage, start=1):
                x = block(x, tracker=tracker)
                if tracker is not None:
                    tracker.log(f"[after Stage{s_idx}.Block{b_idx}]", nn.Identity(), x)

        x = self.gap(x)
        if tracker is not None:
            tracker.log("GlobalAvgPool", self.gap, x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        if tracker is not None:
            tracker.log("Linear(fc)", self.fc, x)
        return x


# --------------------------------------------------------------------------- #
# Verification helpers
# --------------------------------------------------------------------------- #

def verify_block(block: nn.Module, label: str) -> None:
    """Cross-check a block's analytical parameter formula against PyTorch."""
    analytical = block.analytical_param_count()
    actual = count_parameters(block)
    status = "MATCH" if analytical == actual else "MISMATCH"
    print(f"{label:<45} analytical={analytical:>10,}  pytorch={actual:>10,}  [{status}]")
    assert analytical == actual, f"Parameter mismatch in {label}"


def demo_single_blocks() -> None:
    print("=" * 78)
    print("PART A: Single-block structural variants (analytical vs. PyTorch)")
    print("=" * 78)

    x = torch.randn(2, 64, 56, 56)  # (batch, channels, H, W)

    vgg_variants = [
        ("VGGBlock 64->128, 2 convs, BN",  VGGBlock(64, 128, num_convs=2, use_batchnorm=True)),
        ("VGGBlock 64->128, 3 convs, BN",  VGGBlock(64, 128, num_convs=3, use_batchnorm=True)),
        ("VGGBlock 64->128, 2 convs, noBN", VGGBlock(64, 128, num_convs=2, use_batchnorm=False)),
        ("VGGBlock 64->64,  2 convs, k=5",  VGGBlock(64, 64, num_convs=2, kernel_size=5)),
    ]
    for label, block in vgg_variants:
        verify_block(block, label)
        out = block(x)
        print(f"    input {tuple(x.shape)} -> output {tuple(out.shape)}")

    print()
    res_variants = [
        ("ResidualBlock basic 64->64, s=1",       ResidualBlock(64, 64, stride=1, bottleneck=False)),
        ("ResidualBlock basic 64->128, s=2",      ResidualBlock(64, 128, stride=2, bottleneck=False)),
        ("ResidualBlock bottleneck 64->64(x4), s=1", ResidualBlock(64, 64, stride=1, bottleneck=True)),
        ("ResidualBlock bottleneck 256->128(x4), s=2", ResidualBlock(256, 128, stride=2, bottleneck=True)),
    ]
    for label, block in res_variants:
        verify_block(block, label)
        in_ch = block.in_channels
        x_in = torch.randn(2, in_ch, 56, 56)
        out = block(x_in)
        print(f"    input {tuple(x_in.shape)} -> output {tuple(out.shape)}")
    print()


def demo_full_networks() -> None:
    print("=" * 78)
    print("PART B: Full networks assembled from the modular blocks")
    print("=" * 78)

    # --- TinyVGG ---
    vgg_net = TinyVGG(in_channels=3, num_classes=10, base_width=32)
    tracker = ShapeTracker()
    x = torch.randn(1, 3, 224, 224)
    _ = vgg_net(x, tracker=tracker)
    tracker.print_table(title="TinyVGG (VGGBlock x4) -- Layer-by-Layer Trace, input 224x224x3")
    print(f"TinyVGG total parameters (PyTorch count): {count_parameters(vgg_net):,}\n")

    # --- TinyResNet ---
    res_net = TinyResNet(in_channels=3, num_classes=10, base_width=16, bottleneck=True)
    tracker2 = ShapeTracker()
    x2 = torch.randn(1, 3, 224, 224)
    _ = res_net(x2, tracker=tracker2)
    tracker2.print_table(title="TinyResNet (ResidualBlock x8, bottleneck) -- Layer-by-Layer Trace, input 224x224x3")
    print(f"TinyResNet total parameters (PyTorch count): {count_parameters(res_net):,}\n")


if __name__ == "__main__":
    torch.manual_seed(0)
    demo_single_blocks()
    demo_full_networks()

PART A: Single-block structural variants (analytical vs. PyTorch)
VGGBlock 64->128, 2 convs, BN                 analytical=   221,696  pytorch=   221,696  [MATCH]
    input (2, 64, 56, 56) -> output (2, 128, 28, 28)
VGGBlock 64->128, 3 convs, BN                 analytical=   369,408  pytorch=   369,408  [MATCH]
    input (2, 64, 56, 56) -> output (2, 128, 28, 28)
VGGBlock 64->128, 2 convs, noBN               analytical=   221,440  pytorch=   221,440  [MATCH]
    input (2, 64, 56, 56) -> output (2, 128, 28, 28)
VGGBlock 64->64,  2 convs, k=5                analytical=   205,056  pytorch=   205,056  [MATCH]
    input (2, 64, 56, 56) -> output (2, 64, 28, 28)

ResidualBlock basic 64->64, s=1               analytical=    73,984  pytorch=    73,984  [MATCH]
    input (2, 64, 56, 56) -> output (2, 64, 56, 56)
ResidualBlock basic 64->128, s=2              analytical=   230,144  pytorch=   230,144  [MATCH]
    input (2, 64, 56, 56) -> output (2, 128, 28, 28)
ResidualBlock bottleneck 64->64(x4)